This experiment is listed in the [github issue](https://github.com/axelbjarkar/T-404-LOKA-code/issues/1)

In [ ]:
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

import numpy as np

import torch
import torch.nn as nn

import matplotlib.pyplot as plt

# Data preparation
Parse board-state strings into a model-friendly representation (e.g. 5×5 matrix)

In [ ]:
filename = "data/e.txt"
char_map = {'.': 0, 'w': 1, 'b': -1}
outcome_map = {'1':-1, '2':1}

X = []
Y = []

lines = []

# prepare the progress_bar
with open(filename) as f:
    total = sum(1 for _ in f)

# process the lines
with open(filename) as f:
    for line in tqdm(f, total=total, desc='Mapping breakthrough game states'):
        line = line.strip("\n")

        # last shows result
        y = outcome_map[line[-1]]

        # create board
        rows = line.split()[0].split("/")
        board = np.array([[char_map[c] for c in row] for row in rows]).flatten()

        
        X.append(board)
        Y.append(y)

X = np.array(X)
Y = np.array(Y)

# Implement simple MLP

In [ ]:
#from sklearn.neural_network import MLPRegressor

# https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html
#model = MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=500, random_state=SEED)

# was using scikit learn but I'll use pytorch for GPU support in Google Colab
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(25, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze() # (N,1) -> (N)



 # Train and evaluate across different $\alpha$ values

In [ ]:
start = 0.01
end = 0.9
steps = 50

splits = np.linspace(start, end, num=steps, endpoint=True)
mse_values = []
prc_values = []

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

for seed, ratio in enumerate(splits):
    X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=1-ratio, random_state=seed)

    model = MLP().to(device)

    # training setup
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.MSELoss()
    
    # convert data to tensors)
    X_train_t = torch.FloatTensor(X_train).to(device)
    Y_train_t = torch.FloatTensor(Y_train).to(device)
    X_test_t = torch.FloatTensor(X_test).to(device)
    Y_test_t = torch.FloatTensor(Y_test).to(device)
    
    # training loop
    for epoch in tqdm(range(500), desc=f'training model {seed+1}'):
        optimizer.zero_grad()
        predictions = model(X_train_t)
        loss = loss_fn(predictions, Y_train_t)
        loss.backward()
        optimizer.step()
    
    # evaluate
    with torch.no_grad():
        test_preds = model(X_test_t)
        mse = loss_fn(test_preds, Y_test_t).item()
        mse_values.append(mse)
        prc_values.append(ratio)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(np.array(prc_values) * 100, mse_values, 'o-', color='#2196F3', linewidth=2, markersize=8, markerfacecolor='white', markeredgewidth=2)

ax.set_xlabel(r"$\frac{|S|}{|E|}$ (%)", fontsize=14)
ax.set_ylabel("MSE", fontsize=14)
ax.set_title("Model Performance vs Training Subset Size", fontsize=16)

ax.grid(True, alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()